# Gratuity Valuation Project: Stages 1 and 2

**Purpose:** load and check the synthetic employee census that the later valuation stages will use.

**Important:** all data here are synthetic (randomly generated for learning). Nothing describes a real organisation or person.

**Valuation date:** 31 March 2026

## Stage 1, Cell 1: Settings

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

VALUATION_DATE = pd.Timestamp("2026-03-31")
RETIREMENT_AGE = 60

## Stage 1, Cell 2: Load the CSV

Put `synthetic_employee_data.csv` in the **same folder as this notebook** before running this cell.

In [ ]:
df = pd.read_csv(
    "synthetic_employee_data.csv",
    parse_dates=["date_of_birth", "date_of_joining"],
)
df.shape

## Stage 1, Cell 3: First look at the data

In [ ]:
display(df.head())
df.info()

## Stage 1, Cell 4: Derive age and service at the valuation date

Age and service are measured in years, using 365.25 days per year.

In [ ]:
df["age"] = (VALUATION_DATE - df["date_of_birth"]).dt.days / 365.25
df["service_years"] = (VALUATION_DATE - df["date_of_joining"]).dt.days / 365.25
df["age_at_joining"] = df["age"] - df["service_years"]
df[["emp_id", "age", "service_years", "age_at_joining", "monthly_wages_rs"]].head()

## Stage 1, Cell 5: Data quality checks

Run these on every dataset you receive. If a check fails, stop and investigate before valuing anything.

In [ ]:
assert df["emp_id"].is_unique, "Duplicate employee IDs"
assert df.notna().all().all(), "Missing values found"
assert (df["date_of_joining"] <= VALUATION_DATE).all(), "Joined after valuation date"
assert (df["age_at_joining"] >= 21 - 0.01).all(), "Joined before age 21"
assert (df["age"] < RETIREMENT_AGE).all(), "Employee at or past retirement age"
assert (df["monthly_wages_rs"] > 0).all(), "Non-positive wages"
print("All data quality checks passed")

## Stage 1, Cell 6: The part-year rule

Under the Code on Social Security, a part-year of more than six months counts as a full year. We write it as a function and test it, because every benefit calculation in later stages depends on it.

In [ ]:
def completed_years_rounded(service_years):
    """Whole years of service, counting a part-year over six months as a full year."""
    whole = np.floor(service_years)
    part = service_years - whole
    return np.where(part > 0.5, whole + 1, whole)

# Test cases (5 years 5 months, 5 years 7 months, exactly 5 years)
print(completed_years_rounded(np.array([5 + 5/12, 5 + 7/12, 5.0])))   # expect [5. 6. 5.]

df["service_for_formula"] = completed_years_rounded(df["service_years"])

## Stage 1, Cell 7: Statutory benefit if every employee left today (for reference only)

Formula: (15 / 26) x monthly wages x years. This is **not** the DBO. It ignores discounting, future wage growth, exit probabilities and the eligibility rule. We calculate it only to compare with the DBO in later stages. The ceiling is applied; check the current notification and change `CEILING` if it has changed.

In [ ]:
CEILING = 2_000_000   # Rs 20 lakh; verify against the current notification

df["benefit_if_exit_today"] = np.minimum(
    15 / 26 * df["monthly_wages_rs"] * df["service_for_formula"], CEILING
)
df["eligible_today"] = df["service_years"] >= 5   # simplified five-year test

print("Employees:", len(df))
print("Eligible today (>= 5 years):", int(df["eligible_today"].sum()))
print("Total benefit if all eligible employees left today (Rs):",
      int(df.loc[df["eligible_today"], "benefit_if_exit_today"].sum()))

## Stage 1, Cell 8: Summary statistics

In [ ]:
df[["age", "service_years", "monthly_wages_rs"]].describe().round(1)

## Stage 1, Cell 9: Charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(df["age"], bins=15)
axes[0].set_title("Age at valuation date")
axes[0].set_xlabel("Years")

axes[1].hist(df["service_years"], bins=15)
axes[1].set_title("Service at valuation date")
axes[1].set_xlabel("Years")

axes[2].hist(df["monthly_wages_rs"], bins=15)
axes[2].set_title("Monthly wages")
axes[2].set_xlabel("Rs")

plt.tight_layout()
plt.show()

## Stage 1, Cell 10: Your observations (write these yourself)

Write 3 to 4 sentences in your own words: What does the age and service profile look like? How many employees are already eligible? Is the wage distribution skewed, and what could that mean for the liability?

*Your notes here.*

---
# Stage 2: Assumptions and Decrements

**Purpose:** set the valuation assumptions and build the probabilities that an employee leaves service in each future year.

**Important:** these assumptions are **illustrative**. The mortality curve and withdrawal rates are placeholders for learning, not published or company-specific figures. Label them this way in your report.

Stage 2 does not need the CSV; it only defines assumptions. Run all Stage 1 cells first, then these.

## Stage 2, Cell 1: Settings and assumptions

Change values here only. Every later stage reads from this one block, so the whole model updates when you change a number.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ASSUMPTIONS = {
    "valuation_date": pd.Timestamp("2026-03-31"),
    "retirement_age": 60,
    "discount_rate": 0.07,          # ~10-year G-sec yield around the valuation date (check and cite your source)
    "salary_escalation": 0.08,      # illustrative; document the reasoning in your report
    "ceiling": 2_000_000,           # Rs 20 lakh; verify against the current notification
    # Illustrative Gompertz-Makeham mortality: force of mortality = A + B * c**x
    "mort_A": 0.0004,
    "mort_B": 0.00002,
    "mort_c": 1.10,
}

# Illustrative annual withdrawal rates (resignation/termination) by age band: (from_age, to_age_exclusive, rate)
WITHDRAWAL_BANDS = [
    (18, 25, 0.20),
    (25, 30, 0.15),
    (30, 35, 0.11),
    (35, 40, 0.08),
    (40, 45, 0.05),
    (45, 50, 0.03),
    (50, 55, 0.02),
    (55, 60, 0.01),
]

## Stage 2, Cell 2: Mortality

Under a Gompertz-Makeham law the force of mortality is A + B c^x. The probability of dying within one year starting at exact age x is 1 - exp(-integral of the force over that year).

In [ ]:
def mortality_q(x, a=ASSUMPTIONS):
    """One-year probability of death at age x (illustrative Gompertz-Makeham)."""
    A, B, c = a["mort_A"], a["mort_B"], a["mort_c"]
    integral = A + B * c**x * (c - 1) / np.log(c)
    return 1 - np.exp(-integral)

for age in [25, 30, 40, 50, 59]:
    print(f"q at age {age}: {mortality_q(age):.5f}")

## Stage 2, Cell 3: Withdrawal (resignation or termination)

In [ ]:
def withdrawal_w(x, bands=WITHDRAWAL_BANDS):
    """Annual probability of withdrawal at age x, from the age-band table."""
    for lo, hi, rate in bands:
        if lo <= x < hi:
            return rate
    return 0.0

for age in [22, 28, 33, 38, 43, 48, 53, 58]:
    print(f"w at age {age}: {withdrawal_w(age):.2f}")

## Stage 2, Cell 4: Decrement table for one employee

Each year an in-service employee can die, withdraw or stay. We combine the two rates so they do not double count: the death probability is q x (1 - w/2) and the withdrawal probability is w x (1 - q/2). Survivors who reach the retirement age retire at the end of the last year.

Simplification: age is rounded down to whole years and all exits are treated as happening at the end of the year.

In [ ]:
def exit_table(age_now, a=ASSUMPTIONS):
    """Probability of each type of exit in each future year for an employee aged age_now."""
    ret = a["retirement_age"]
    start_age = int(np.floor(age_now))
    rows = []
    in_service = 1.0                       # probability still in service at the start of the year
    for t in range(1, ret - start_age + 1):
        x = start_age + t - 1              # age at the start of year t
        q = mortality_q(x)
        w = withdrawal_w(x)
        p_death = in_service * q * (1 - w / 2)
        p_withdraw = in_service * w * (1 - q / 2)
        p_stay = in_service - p_death - p_withdraw
        p_retire = p_stay if t == ret - start_age else 0.0
        rows.append({
            "t": t,
            "age_at_start": x,
            "in_service_start": in_service,
            "p_death": p_death,
            "p_withdraw": p_withdraw,
            "p_retire": p_retire,
        })
        in_service = p_stay
    return pd.DataFrame(rows)

tbl = exit_table(35)
tbl.round(4).head(10)

## Stage 2, Cell 5: Check that probabilities add up to 1

If the total of all exit probabilities is not exactly 1, the table is wrong and every later number will be wrong too.

In [ ]:
total = tbl[["p_death", "p_withdraw", "p_retire"]].sum().sum()
print("Sum of all exit probabilities:", round(total, 10))
assert abs(total - 1) < 1e-9
print("Check passed")

print("Probability of ending service by death     :", round(tbl.p_death.sum(), 4))
print("Probability of ending service by withdrawal:", round(tbl.p_withdraw.sum(), 4))
print("Probability of reaching retirement          :", round(tbl.p_retire.sum(), 4))

## Stage 2, Cell 6: Chart of exits by year for an employee aged 35

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(tbl["t"], tbl["p_withdraw"], label="Withdrawal")
ax.bar(tbl["t"], tbl["p_death"], bottom=tbl["p_withdraw"], label="Death")
ax.bar(tbl["t"], tbl["p_retire"], bottom=tbl["p_withdraw"] + tbl["p_death"], label="Retirement")
ax.set_xlabel("Years from valuation date")
ax.set_ylabel("Probability of exit in that year")
ax.set_title("Exit probabilities for an employee aged 35 (illustrative assumptions)")
ax.legend()
plt.tight_layout()
plt.show()

## Stage 2, Cell 7: Compare ages

Run the table for three different ages and compare. This shows why the withdrawal assumption matters more for young employees.

In [ ]:
rows = []
for age in [25, 35, 45, 55]:
    t = exit_table(age)
    rows.append({
        "age": age,
        "years_to_retirement": ASSUMPTIONS["retirement_age"] - age,
        "P(withdraw)": round(t.p_withdraw.sum(), 3),
        "P(death)": round(t.p_death.sum(), 4),
        "P(reach retirement)": round(t.p_retire.sum(), 3),
    })
pd.DataFrame(rows)

## Stage 2, Cell 8: Your observations (write these yourself)

Look at the chart and the comparison table. In 3 to 4 sentences: In which years is exit most likely for a 35-year-old, and why? How does the chance of reaching retirement change with age? What does this mean for how much of the liability is paid early versus late?

*Your notes here.*

# Stage 3: The Valuation Engine (Projected Unit Credit)

**Run this after Stage 1 and Stage 2** in the same notebook. It uses `df`, `ASSUMPTIONS`, `exit_table()` and `completed_years_rounded()` from the earlier stages.

**What it calculates for each employee:** the expected gratuity payable in each future year, the share of it earned so far (attributed by service), and the present value of that share. The sum across employees is the DBO.

**Simplifications (state these in your report):**
- Ages are rounded down to whole years and all exits occur at the end of a year.
- The ceiling is applied to the projected benefit, then the benefit is attributed in proportion to service.
- Death is paid regardless of service length. Withdrawal is paid only if service at exit is 5 years or more. Retirement is paid if service is 5 years or more (always true for the cases here).
- Interest cost is approximated as discount rate x opening DBO.

## Stage 3, Cell 1: The valuation function for one employee

In [ ]:
def value_employee(age_now, service_now, wage, a=ASSUMPTIONS, table=None):
    """Projected Unit Credit valuation of gratuity for one employee.
    Returns DBO, current service cost and the expected payment by future year."""
    if table is None:
        table = exit_table(age_now, a)

    t = table["t"].to_numpy()
    v = 1 / (1 + a["discount_rate"])

    service_t = service_now + t                                   # service at exit
    wage_t = wage * (1 + a["salary_escalation"]) ** t             # projected monthly wages at exit
    years_t = completed_years_rounded(service_t)                  # part-year rule
    benefit_t = np.minimum(15 / 26 * wage_t * years_t, a["ceiling"])

    withdrawal_paid = (service_t >= 5).astype(float)              # five-year rule for withdrawals
    prob_paid = table["p_death"].to_numpy() + table["p_retire"].to_numpy() \
                + table["p_withdraw"].to_numpy() * withdrawal_paid

    expected_benefit = prob_paid * benefit_t                      # expected payment at the end of year t
    attributed = expected_benefit * service_now / service_t       # share earned to date (PUC)
    disc = v ** t

    return {
        "dbo": float((attributed * disc).sum()),
        "service_cost": float((expected_benefit / service_t * disc).sum()),
        "attributed_by_year": attributed,
        "detail": pd.DataFrame({
            "t": t, "service_at_exit": service_t.round(2), "wage_at_exit": wage_t.round(0),
            "years_for_formula": years_t, "benefit_if_paid": benefit_t.round(0),
            "prob_paid": prob_paid.round(4), "expected_benefit": expected_benefit.round(0),
            "share_earned": (service_now / service_t).round(3), "discount_factor": disc.round(4),
            "pv_attributed": (attributed * disc).round(0),
        }),
    }

## Stage 3, Cell 2: Validate against the hand example from the theory lesson

Before trusting the function on 150 employees, test it on a case we can calculate by hand: age 40, 10 years of service, wages Rs 60,000, retires at 60 with certainty, wage growth 6%, discount rate 7%. By hand: DBO = Rs 1,72,279 and service cost = Rs 17,228.

In [ ]:
a_test = {**ASSUMPTIONS, "discount_rate": 0.07, "salary_escalation": 0.06}

# A table where the employee retires with certainty at t = 20 and nothing else happens
test_table = pd.DataFrame({"t": range(1, 21)})
test_table["p_death"] = 0.0
test_table["p_withdraw"] = 0.0
test_table["p_retire"] = 0.0
test_table.loc[test_table["t"] == 20, "p_retire"] = 1.0

res = value_employee(age_now=40, service_now=10, wage=60000, a=a_test, table=test_table)
print("DBO         :", round(res["dbo"]))
print("Service cost:", round(res["service_cost"]))

assert abs(res["dbo"] - 172279) < 5, "DBO does not match the hand calculation"
assert abs(res["service_cost"] - 17228) < 2, "Service cost does not match the hand calculation"
print("Validation passed: the function reproduces the hand calculation")

## Stage 3, Cell 3: Follow one employee's calculation line by line

Pick any employee and read the working. This is the table to explain in an interview.

In [ ]:
emp = df[df["emp_id"] == "E003"].iloc[0]
r = value_employee(emp["age"], emp["service_years"], emp["monthly_wages_rs"])
print(f"E003: age {emp['age']:.1f}, service {emp['service_years']:.1f} years, wages Rs {emp['monthly_wages_rs']:,}")
print(f"DBO: Rs {r['dbo']:,.0f}   Service cost: Rs {r['service_cost']:,.0f}")
r["detail"].head(8)

## Stage 3, Cell 4: Value all employees

In [ ]:
max_t = ASSUMPTIONS["retirement_age"] - int(np.floor(df["age"].min()))
cash_by_year = np.zeros(max_t)          # expected payments for past service, by year, summed over employees

rows = []
for _, e in df.iterrows():
    r = value_employee(e["age"], e["service_years"], e["monthly_wages_rs"])
    cash_by_year[: len(r["attributed_by_year"])] += r["attributed_by_year"]
    rows.append({"emp_id": e["emp_id"], "age": e["age"], "service_years": e["service_years"],
                 "monthly_wages_rs": e["monthly_wages_rs"], "dbo": r["dbo"], "service_cost": r["service_cost"]})

results = pd.DataFrame(rows)

total_dbo = results["dbo"].sum()
total_sc = results["service_cost"].sum()
interest_cost = ASSUMPTIONS["discount_rate"] * total_dbo

print(f"Total DBO at valuation date    : Rs {total_dbo:,.0f}  (Rs {total_dbo/1e7:.2f} crore)")
print(f"Current service cost (next yr) : Rs {total_sc:,.0f}")
print(f"Interest cost (approx.)        : Rs {interest_cost:,.0f}")
print(f"Expected benefit payments, year 1 (past service): Rs {cash_by_year[0]:,.0f}")

## Stage 3, Cell 5: Checks

Two independent checks that the numbers hang together.

In [ ]:
# Check 1: discounting the cash-flow projection must give back the DBO
v = 1 / (1 + ASSUMPTIONS["discount_rate"])
pv_from_cash = (cash_by_year * v ** np.arange(1, max_t + 1)).sum()
print("DBO from employee table :", round(total_dbo))
print("DBO from cash-flow table:", round(pv_from_cash))
assert abs(total_dbo - pv_from_cash) < 1, "Cash-flow projection does not reconcile to the DBO"

# Check 2: no negative values, and every employee has a positive DBO
assert (results["dbo"] >= 0).all()
print("All checks passed")

# Weighted average time to payment (duration) of the liability
years = np.arange(1, max_t + 1)
pv_by_year = cash_by_year * v ** years
duration = (years * pv_by_year).sum() / pv_by_year.sum()
print(f"Weighted average term of the liability: {duration:.1f} years")

## Stage 3, Cell 6: Compare with the Stage 1 reference figure

In [ ]:
reference = df.loc[df["eligible_today"], "benefit_if_exit_today"].sum()
print(f"Benefit if all eligible employees left today (Stage 1): Rs {reference:,.0f}")
print(f"DBO from the valuation                               : Rs {total_dbo:,.0f}")
print(f"DBO as a share of the Stage 1 reference figure       : {total_dbo / reference:.1%}")

## Stage 3, Cell 7: Where the liability sits

In [ ]:
top10 = results.sort_values("dbo", ascending=False).head(10)
print(f"Top 10 employees hold {top10['dbo'].sum() / total_dbo:.1%} of the DBO")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(range(1, max_t + 1), cash_by_year / 1e5)
axes[0].set_xlabel("Years from valuation date")
axes[0].set_ylabel("Rs lakh")
axes[0].set_title("Expected payments for service earned to date (undiscounted)")

bands = pd.cut(results["age"], bins=[20, 30, 40, 50, 60], right=False,
               labels=["22-29", "30-39", "40-49", "50-59"])
axes[1].bar(results.groupby(bands, observed=True)["dbo"].sum().index.astype(str),
            results.groupby(bands, observed=True)["dbo"].sum() / 1e5)
axes[1].set_xlabel("Age band")
axes[1].set_ylabel("DBO, Rs lakh")
axes[1].set_title("DBO by age band")

plt.tight_layout()
plt.show()

## Stage 3, Cell 8: Your observations (write these yourself)

In 4 to 5 sentences: What is the total DBO and how does it compare with the Stage 1 reference figure? Why is it smaller or larger? When are most payments expected, and what does the weighted average term tell you? Which age bands hold most of the liability?

*Your notes here.*